In [4]:
# ============================================================
# SolarDW - ETL Pipeline (Extract → Transform → Load)
# Data Warehouse for Solar Power Generation
# ============================================================

# %% [1] الاستيراد
import pandas as pd
import numpy as np
import mysql.connector

print("=" * 60)
print("STARTING ETL PIPELINE")
print("=" * 60)

# ============================================================
# PHASE 1: EXTRACT – قراءة ملفات CSV
# ============================================================

# %% [2] قراءة ملفات البيانات
try:
    generation_df = pd.read_csv("Plant_1_Generation_Data.csv")
    print(f"✅ Generation Data Loaded: {generation_df.shape}")
    print(f"   Columns: {list(generation_df.columns)}")
except FileNotFoundError:
    print("❌ Plant_1_Generation_Data.csv not found!")
    exit(1)

try:
    weather_sensor_df = pd.read_csv("Plant_1_Weather_Sensor_Data.csv")
    print(f"✅ Weather Sensor Data Loaded: {weather_sensor_df.shape}")
    print(f"   Columns: {list(weather_sensor_df.columns)}")
except FileNotFoundError:
    print("❌ Plant_1_Weather_Sensor_Data.csv not found!")
    exit(1)

try:
    meteo_2019_df = pd.read_csv("2019.csv")
    print(f"✅ External 2019 Data Loaded: {meteo_2019_df.shape}")
    print(f"   Columns: {list(meteo_2019_df.columns)[:5]}...")
except FileNotFoundError:
    print("❌ 2019.csv not found!")
    exit(1)

# ============================================================
# PHASE 2: TRANSFORM – تنظيف وتحويل البيانات
# ============================================================

# %% [3] تجهيز بيانات التوليد
# تحويل DATE_TIME
generation_df['DATE_TIME'] = pd.to_datetime(
    generation_df['DATE_TIME'],
    format="%d-%m-%Y %H:%M",
    errors='coerce'
)

# حذف الصفوف التالفة
generation_df = generation_df.dropna(subset=['DATE_TIME'])

# استخراج PLANT_ID من البيانات (كما هو موجود في الملف)
# البيانات الأصلية: PLANT_ID = 4135001 (رقم)
generation_df['PLANT_ID'] = generation_df['PLANT_ID'].astype(int)

# إنشاء مفتاح التاريخ
generation_df['date_key'] = (
    generation_df['DATE_TIME']
    .dt.strftime("%Y%m%d%H%M")
)

print(f"\n✅ Generation Data Cleaned: {generation_df.shape}")
print(f"   Unique PLANT_ID in generation: {generation_df['PLANT_ID'].unique()}")
print(f"   Date range: {generation_df['DATE_TIME'].min()} to {generation_df['DATE_TIME'].max()}")

# ------------------------------------------------------------

# %% [4] تجهيز بيانات حساس الطقس
# تحويل DATE_TIME
weather_sensor_df['DATE_TIME'] = pd.to_datetime(
    weather_sensor_df['DATE_TIME'],
    errors='coerce'
)

# حذف الصفوف التالفة
weather_sensor_df = weather_sensor_df.dropna(subset=['DATE_TIME'])

# استخراج PLANT_ID من بيانات الطقس
weather_sensor_df['PLANT_ID'] = weather_sensor_df['PLANT_ID'].astype(int)

# إنشاء مفتاح التاريخ
weather_sensor_df['date_key'] = (
    weather_sensor_df['DATE_TIME']
    .dt.strftime("%Y%m%d%H%M")
)

print(f"\n✅ Weather Sensor Data Cleaned: {weather_sensor_df.shape}")
print(f"   Unique PLANT_ID in weather: {weather_sensor_df['PLANT_ID'].unique()}")
print(f"   Date range: {weather_sensor_df['DATE_TIME'].min()} to {weather_sensor_df['DATE_TIME'].max()}")

# ------------------------------------------------------------

# %% [5] تجهيز بيانات الطقس الخارجية
# تحويل datetime
meteo_2019_df['datetime'] = pd.to_datetime(
    meteo_2019_df['datetime'],
    errors='coerce'
)

# حذف الصفوف التالفة
meteo_2019_df = meteo_2019_df.dropna(subset=['datetime'])

# إنشاء مفتاح التاريخ
meteo_2019_df['date_key'] = (
    meteo_2019_df['datetime']
    .dt.strftime("%Y%m%d%H%M")
)

# تعديل أسماء الأعمدة لتوافق MySQL
meteo_2019_df.rename(columns={
    "Relative Humidity": "Relative_Humidity",
    "Wind Speed": "Wind_Speed"
}, inplace=True)

print(f"\n✅ External Weather Data Cleaned: {meteo_2019_df.shape}")
print(f"   Date range: {meteo_2019_df['datetime'].min()} to {meteo_2019_df['datetime'].max()}")

# ============================================================
# PHASE 3: BUILD DIM_DATE – من جميع المصادر
# ============================================================

# %% [6] جمع كل التواريخ من جميع المصادر
all_dates_list = []

# من بيانات التوليد
all_dates_list.append(
    generation_df[['DATE_TIME', 'date_key']]
)

# من بيانات حساس الطقس
all_dates_list.append(
    weather_sensor_df[['DATE_TIME', 'date_key']]
)

# من بيانات الطقس الخارجي
meteo_dates = meteo_2019_df[['datetime', 'date_key']].copy()
meteo_dates.rename(columns={'datetime': 'DATE_TIME'}, inplace=True)
all_dates_list.append(meteo_dates)

# دمج جميع التواريخ
all_dates = pd.concat(all_dates_list, ignore_index=True)

# إزالة المكررات
all_dates = all_dates.drop_duplicates(subset=['date_key'])

# حذف القيم الفارغة
all_dates = all_dates.dropna(subset=['DATE_TIME'])

print(f"\n✅ Total unique dates from all sources: {len(all_dates)}")

# ------------------------------------------------------------

# بناء DIM_DATE
dim_date = all_dates.copy()

dim_date['full_date'] = pd.to_datetime(dim_date['DATE_TIME']).dt.date
dim_date['day'] = pd.to_datetime(dim_date['DATE_TIME']).dt.day
dim_date['month'] = pd.to_datetime(dim_date['DATE_TIME']).dt.month
dim_date['year'] = pd.to_datetime(dim_date['DATE_TIME']).dt.year
dim_date['quarter'] = pd.to_datetime(dim_date['DATE_TIME']).dt.quarter
dim_date['hour'] = pd.to_datetime(dim_date['DATE_TIME']).dt.hour
dim_date['weekday'] = pd.to_datetime(dim_date['DATE_TIME']).dt.day_name()

# ترتيب الأعمدة
dim_date = dim_date[[
    'date_key', 'full_date', 'day', 'month',
    'year', 'quarter', 'hour', 'weekday'
]]

print(f"✅ DIM_DATE Created: {dim_date.shape[0]:,} rows")
print(f"   Date range: {dim_date['full_date'].min()} to {dim_date['full_date'].max()}")

# ============================================================
# PHASE 4: BUILD DIM_PLANT (من generation و weather معاً)
# ============================================================

# %% [7] بناء جدول المحطات من جميع المصادر
# جمع كل المحطات من المصادر المختلفة
all_plants_list = []

# من بيانات التوليد
all_plants_list.append(generation_df[['PLANT_ID']])

# من بيانات حساس الطقس
all_plants_list.append(weather_sensor_df[['PLANT_ID']])

# دمج جميع المحطات
all_plants = pd.concat(all_plants_list, ignore_index=True)

# إزالة المكررات
all_plants = all_plants.drop_duplicates().reset_index(drop=True)

# إنشاء surrogate key
all_plants['plant_key'] = all_plants.index + 1

# إنشاء اسم للمحطة
all_plants['plant_name'] = "Plant_" + all_plants['PLANT_ID'].astype(str)

# ترتيب الأعمدة
dim_plant = all_plants[['plant_key', 'PLANT_ID', 'plant_name']]

print(f"\n✅ DIM_PLANT Created: {dim_plant.shape[0]} rows")
print(dim_plant)

# ============================================================
# PHASE 5: BUILD DIM_SOURCE
# ============================================================

# %% [8] بناء جدول الإنفرترات
dim_source = (
    generation_df[['SOURCE_KEY']]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

# إنشاء surrogate key
dim_source['source_key'] = dim_source.index + 1

# إعادة تسمية العمود
dim_source.rename(columns={'SOURCE_KEY': 'SOURCE_ID'}, inplace=True)

# ترتيب الأعمدة
dim_source = dim_source[['source_key', 'SOURCE_ID']]

print(f"\n✅ DIM_SOURCE Created: {dim_source.shape[0]} rows")
print(dim_source.head())

# ============================================================
# PHASE 6: ENRICH – إضافة المفاتيح البديلة
# ============================================================

# %% [9] إضافة plant_key إلى generation_df
generation_df = generation_df.merge(
    dim_plant[['plant_key', 'PLANT_ID']],
    on='PLANT_ID',
    how='left'
)

# إضافة source_key إلى generation_df
generation_df = generation_df.merge(
    dim_source,
    left_on='SOURCE_KEY',
    right_on='SOURCE_ID',
    how='left'
)

# حذف العمود المكرر
generation_df.drop(columns=['SOURCE_ID'], inplace=True)

print("\n✅ Generation Data Enriched with keys")
print(f"   plant_key sample: {generation_df['plant_key'].iloc[0]}")
print(f"   source_key sample: {generation_df['source_key'].iloc[0]}")

# ============================================================
# PHASE 7: MERGE – دمج البيانات
# ============================================================

# %% [10] دمج بيانات التوليد مع حساس الطقس
merged_df = generation_df.merge(
    weather_sensor_df,
    on=['date_key', 'PLANT_ID'],
    how='left'
)

print(f"\n✅ After merging with weather sensor: {merged_df.shape}")

# دمج بيانات الطقس الخارجية
merged_df = merged_df.merge(
    meteo_2019_df[[
        'date_key', 'Temperature', 'DHI', 'DNI', 'GHI',
        'Relative_Humidity', 'Pressure', 'Wind_Speed'
    ]],
    on='date_key',
    how='left'
)

print(f"✅ After merging with external weather: {merged_df.shape}")

# فحص التكرارات
duplicates = merged_df.duplicated().sum()
print(f"✅ Duplicate Rows: {duplicates}")

# فحص القيم المفقودة
print(f"\n📊 Missing Values Check:")
print(f"   DC_POWER missing: {merged_df['DC_POWER'].isna().sum():,}")
print(f"   AC_POWER missing: {merged_df['AC_POWER'].isna().sum():,}")
print(f"   Temperature missing: {merged_df['Temperature'].isna().sum():,}")

# ============================================================
# PHASE 8: SAMPLE DATA (اختياري)
# ============================================================

# %% [11] أخذ عينة من البيانات (اختر الحجم المناسب)
SAMPLE_SIZE = 5000  # غيّر هذا الرقم حسب حاجتك

if len(merged_df) > SAMPLE_SIZE:
    merged_df = merged_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"\n✅ Sample Applied: {merged_df.shape} rows")
else:
    print(f"\n✅ Using all data: {merged_df.shape} rows")

# ============================================================
# PHASE 9: BUILD FACT TABLE
# ============================================================

# %% [12] إنشاء FACT TABLE
fact_df = merged_df[[
    'date_key', 'source_key', 'plant_key',
    'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD',
    'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION',
    'Temperature', 'DHI', 'DNI', 'GHI',
    'Relative_Humidity', 'Pressure', 'Wind_Speed'
]].copy()

# حذف التكرارات
fact_df = fact_df.drop_duplicates()

# إعادة ضبط الفهرس
fact_df = fact_df.reset_index(drop=True)

# إنشاء fact_id
fact_df.insert(0, 'fact_id', range(1, len(fact_df) + 1))

# تحويل NaN إلى None (لـ MySQL)
fact_df = fact_df.replace({np.nan: None, "nan": None, "NaN": None, "": None})

print(f"\n✅ FACT TABLE Created: {fact_df.shape}")
print(f"   Columns: {list(fact_df.columns)}")

# ============================================================
# PHASE 10: CONNECT TO MYSQL
# ============================================================

# %% [13] الاتصال بقاعدة البيانات
try:
    # إنشاء قاعدة البيانات إذا لم تكن موجودة
    temp_conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="R123456"
    )
    temp_cursor = temp_conn.cursor()
    temp_cursor.execute("CREATE DATABASE IF NOT EXISTS SolarDW")
    temp_cursor.close()
    temp_conn.close()
    print("\n✅ Database 'solardw' ready")

    # الاتصال بقاعدة البيانات
    connection = mysql.connector.connect(
        host="localhost",
        user="root",
        password="R123456",
        database="solardw"
    )
    cursor = connection.cursor()
    print("✅ Connected to MySQL")

except Exception as e:
    print(f"\n❌ MySQL Error: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. تأكد من تشغيل MySQL (XAMPP/WAMP/MAMP)")
    print("   2. تأكد من صحة كلمة المرور")
    print("   3. جرب: sudo mysql (على Linux/Mac)")
    exit(1)

# ============================================================
# PHASE 11: CREATE TABLES
# ============================================================

# %% [14] حذف الجداول القديمة
cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
cursor.execute("DROP TABLE IF EXISTS fact_power_generation")
cursor.execute("DROP TABLE IF EXISTS dim_date")
cursor.execute("DROP TABLE IF EXISTS dim_plant")
cursor.execute("DROP TABLE IF EXISTS dim_source")
cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
print("\n✅ Old tables dropped")

# ------------------------------------------------------------
# إنشاء جدول DIM_DATE
cursor.execute("""
CREATE TABLE dim_date (
    date_key VARCHAR(12) PRIMARY KEY,
    full_date DATE,
    day INT,
    month INT,
    year INT,
    quarter INT,
    hour INT,
    weekday VARCHAR(20)
)
""")

# ------------------------------------------------------------
# إنشاء جدول DIM_PLANT
cursor.execute("""
CREATE TABLE dim_plant (
    plant_key INT PRIMARY KEY,
    PLANT_ID INT,
    plant_name VARCHAR(100)
)
""")

# ------------------------------------------------------------
# إنشاء جدول DIM_SOURCE
cursor.execute("""
CREATE TABLE dim_source (
    source_key INT PRIMARY KEY,
    SOURCE_ID VARCHAR(50)
)
""")

# ------------------------------------------------------------
# إنشاء FACT TABLE
cursor.execute("""
CREATE TABLE fact_power_generation (
    fact_id INT PRIMARY KEY,
    date_key VARCHAR(12),
    source_key INT,
    plant_key INT,
    DC_POWER FLOAT,
    AC_POWER FLOAT,
    DAILY_YIELD FLOAT,
    TOTAL_YIELD FLOAT,
    AMBIENT_TEMPERATURE FLOAT,
    MODULE_TEMPERATURE FLOAT,
    IRRADIATION FLOAT,
    Temperature FLOAT,
    DHI FLOAT,
    DNI FLOAT,
    GHI FLOAT,
    Relative_Humidity FLOAT,
    Pressure FLOAT,
    Wind_Speed FLOAT,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (source_key) REFERENCES dim_source(source_key),
    FOREIGN KEY (plant_key) REFERENCES dim_plant(plant_key)
)
""")

print("✅ All tables created")

# ============================================================
# PHASE 12: LOAD DIMENSIONS
# ============================================================

# %% [15] تحميل DIM_DATE
rows_inserted = 0
for _, row in dim_date.iterrows():
    try:
        cursor.execute("""
            INSERT INTO dim_date (
                date_key, full_date, day, month, year, quarter, hour, weekday
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            str(row['date_key']), row['full_date'],
            int(row['day']), int(row['month']),
            int(row['year']), int(row['quarter']),
            int(row['hour']), str(row['weekday'])
        ))
        rows_inserted += 1
    except Exception as e:
        print(f"   ⚠️ Error inserting date {row['date_key']}: {e}")

connection.commit()
print(f"\n✅ DIM_DATE Loaded: {rows_inserted:,} rows")

# ------------------------------------------------------------

# %% [16] تحميل DIM_PLANT
rows_inserted = 0
for _, row in dim_plant.iterrows():
    try:
        cursor.execute("""
            INSERT INTO dim_plant (plant_key, PLANT_ID, plant_name)
            VALUES (%s, %s, %s)
        """, (
            int(row['plant_key']),
            int(row['PLANT_ID']),
            str(row['plant_name'])
        ))
        rows_inserted += 1
    except Exception as e:
        print(f"   ⚠️ Error: {e}")

connection.commit()
print(f"✅ DIM_PLANT Loaded: {rows_inserted} rows")

# ------------------------------------------------------------

# %% [17] تحميل DIM_SOURCE
rows_inserted = 0
for _, row in dim_source.iterrows():
    try:
        cursor.execute("""
            INSERT INTO dim_source (source_key, SOURCE_ID)
            VALUES (%s, %s)
        """, (
            int(row['source_key']),
            str(row['SOURCE_ID'])
        ))
        rows_inserted += 1
    except Exception as e:
        print(f"   ⚠️ Error: {e}")

connection.commit()
print(f"✅ DIM_SOURCE Loaded: {rows_inserted} rows")

# ============================================================
# PHASE 13: LOAD FACT TABLE
# ============================================================

# %% [18] تحميل FACT TABLE
errors = 0
rows_inserted = 0

for i, row in fact_df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO fact_power_generation (
                fact_id, date_key, source_key, plant_key,
                DC_POWER, AC_POWER, DAILY_YIELD, TOTAL_YIELD,
                AMBIENT_TEMPERATURE, MODULE_TEMPERATURE, IRRADIATION,
                Temperature, DHI, DNI, GHI,
                Relative_Humidity, Pressure, Wind_Speed
            ) VALUES (
                %s, %s, %s, %s, %s, %s, %s, %s,
                %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
            )
        """, (
            int(row['fact_id']),
            str(row['date_key']),
            int(row['source_key']) if row['source_key'] is not None else None,
            int(row['plant_key']) if row['plant_key'] is not None else None,
            row['DC_POWER'], row['AC_POWER'], row['DAILY_YIELD'], row['TOTAL_YIELD'],
            row['AMBIENT_TEMPERATURE'], row['MODULE_TEMPERATURE'], row['IRRADIATION'],
            row['Temperature'], row['DHI'], row['DNI'], row['GHI'],
            row['Relative_Humidity'], row['Pressure'], row['Wind_Speed']
        ))
        rows_inserted += 1

        # عرض التقدم كل 500 صف
        if rows_inserted % 500 == 0:
            print(f"   Progress: {rows_inserted:,} rows inserted...")

    except Exception as e:
        errors += 1
        if errors <= 5:
            print(f"   ❌ Error at row {i}: {e}")
        if errors >= 10:
            print("   Too many errors - stopping")
            break

connection.commit()
print(f"\n✅ FACT TABLE Loaded: {rows_inserted:,} rows inserted, {errors} errors")

# ============================================================
# PHASE 14: VERIFICATION (اختياري)
# ============================================================

# %% [19] التحقق من البيانات
print("\n📊 Verification:")

cursor.execute("SELECT COUNT(*) FROM dim_date")
dim_date_count = cursor.fetchone()[0]
print(f"   dim_date count: {dim_date_count:,}")

cursor.execute("SELECT COUNT(*) FROM dim_plant")
dim_plant_count = cursor.fetchone()[0]
print(f"   dim_plant count: {dim_plant_count}")

cursor.execute("SELECT COUNT(*) FROM dim_source")
dim_source_count = cursor.fetchone()[0]
print(f"   dim_source count: {dim_source_count}")

cursor.execute("SELECT COUNT(*) FROM fact_power_generation")
fact_count = cursor.fetchone()[0]
print(f"   fact_power_generation count: {fact_count:,}")

# ============================================================
# PHASE 15: CLOSE CONNECTION
# ============================================================

# %% [20] إغلاق الاتصال
cursor.close()
connection.close()

print("\n" + "=" * 60)
print("🎉 ETL PIPELINE COMPLETED SUCCESSFULLY 🎉")
print("=" * 60)

print("\n📊 FINAL SUMMARY")
print(f"   DIM_DATE   : {len(dim_date):,} rows")
print(f"   DIM_PLANT  : {len(dim_plant):,} rows")
print(f"   DIM_SOURCE : {len(dim_source):,} rows")
print(f"   FACT_TABLE : {rows_inserted:,} rows")

print("\n✅ SolarDW is ready for Power BI / Analytics")
print("=" * 60)

STARTING ETL PIPELINE
✅ Generation Data Loaded: (68778, 8)
   Columns: ['PK_ID', 'DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']
✅ Weather Sensor Data Loaded: (3182, 7)
   Columns: ['PK_ID', 'DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
✅ External 2019 Data Loaded: (35040, 15)
   Columns: ['PK_ID', 'Temperature', 'Clearsky DHI', 'Clearsky DNI', 'Clearsky GHI']...

✅ Generation Data Cleaned: (68778, 9)
   Unique PLANT_ID in generation: [4135001]
   Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00

✅ Weather Sensor Data Cleaned: (3182, 8)
   Unique PLANT_ID in weather: [4135001]
   Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00

✅ External Weather Data Cleaned: (35040, 16)
   Date range: 2019-01-01 00:00:00 to 2019-12-31 23:45:00

✅ Total unique dates from all sources: 38223
✅ DIM_DATE Created: 38,223 rows
   Date range: 2019-01-01 to 2020-06-17

✅ DIM_PLANT Created: 1 rows
   